# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zarnabrajpoot956-web/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. What one row means for my lane: one page, on one day (page-day grain).
2. Which table(s): [fill in once we see real table names from Step 4]
3. Time window: month=2026-03 (mid-panel, avoiding the sealed final month).
4. What I'd predict/rank: ctr_gap how far a page's CTR sits below
   similar-position pages, used to rank a review queue.
5. What I exclude: any column reflecting outcomes AFTER the decision
   point (future traffic, later edits) that would leak the label.

In [27]:
from huggingface_hub import hf_hub_download
import pandas as pd

performance_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

df = pd.read_parquet(performance_file)

print("Columns:")
print(df.columns.tolist())

print("\nShape:")
print(df.shape)

print("\nFirst 5 rows:")
display(df.head())

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']

Shape:
(9841378, 30)

First 5 rows:


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
# Query 1: Check page-day grain
grain_check = df.groupby(['content_hash_id', 'report_date']).size()

print("Duplicate page-days:", (grain_check > 1).sum())


# Query 2: Row count and date range
print("Total rows:", len(df))
print("Date range:", df['report_date'].min(), "to", df['report_date'].max())


# Query 3: Check available GSC data
available = df[df['gsc_data_available'] == True]

print("Rows before availability filter:", len(df))
print("Rows after GSC availability filter:", len(available))

Duplicate page-days: 0
Total rows: 9841378
Date range: 2026-03-01 to 2026-03-31
Rows before availability filter: 9841378
Rows after GSC availability filter: 3611061


In [29]:
print(df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [30]:
import numpy as np
# Use rows where GSC data is available
features = df[df['gsc_data_available'] == True].copy()

# Feature 1: CTR
features['ctr'] = (
    features['gsc_clicks'] / features['gsc_impressions'].replace(0, np.nan)
)

# Feature 2: Average position
features['avg_position'] = features['gsc_avg_position']

# Feature 3: 7-day impressions average
features['impressions_7d_avg'] = (
    features.groupby(['client_hash_id', 'content_hash_id'])['gsc_impressions']
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

# Feature 4: Engagement per session
features['engagement_per_session'] = (
    features['ga4_total_engagement_sec'] /
    features['ga4_sessions'].replace(0, np.nan)
)

# Feature 5: Scroll events per session
features['scroll_per_session'] = (
    features['scroll_events'] /
    features['ga4_sessions'].replace(0, np.nan)
)

print("Five features created:")
print([
    'ctr',
    'avg_position',
    'impressions_7d_avg',
    'engagement_per_session',
    'scroll_per_session'
])

Five features created:
['ctr', 'avg_position', 'impressions_7d_avg', 'engagement_per_session', 'scroll_per_session']


In [35]:
# DELIBERATE LABEL LEAKAGE
features['LEAKY_future_ctr'] = (
    features.groupby(['client_hash_id', 'content_hash_id'])['ctr']
    .shift(-7)
)

print("Leaky feature created successfully.")

Leaky feature created successfully.


In [36]:
# Show the future values being used
print(features[['report_date', 'content_hash_id', 'ctr', 'LEAKY_future_ctr']].head(15))

   report_date           content_hash_id       ctr  LEAKY_future_ctr
0   2026-03-01  content_b7e512995f79d5a6  0.000000          0.000000
1   2026-03-01  content_05597932fe4da067  0.000000          0.000000
2   2026-03-01  content_7a105f548d9c6916  0.008000          0.000000
3   2026-03-01  content_905aa32a0230694e  0.000000          0.000000
4   2026-03-01  content_a3ea9792f793ec72  0.000000          0.000000
5   2026-03-01  content_36c36abc7650d7af  0.004184          0.000000
6   2026-03-01  content_a7da352b73b02668  0.000000          0.004219
7   2026-03-01  content_05434271b257bb68  0.000000          0.000000
8   2026-03-01  content_d056587ff7faca0c  0.000000          0.000000
9   2026-03-01  content_bfd1e41c2af250c8  0.000000          0.000000
10  2026-03-01  content_2662845f598544ef  0.000000          0.000000
12  2026-03-01  content_1855a661b4d36130  0.000000          0.000000
13  2026-03-01  content_22610b0934f8825e  0.000000          0.000000
14  2026-03-01  content_712c365258

In [37]:
# Compare current CTR with future CTR
leakage_corr = features[['ctr', 'LEAKY_future_ctr']].corr().iloc[0, 1]

print("Correlation with future CTR:", leakage_corr)

Correlation with future CTR: 0.03349601003108989


In [39]:
# Remove the deliberately leaky feature
features_clean = features.drop(columns=['LEAKY_future_ctr'])

print("Leaky feature removed.")
print("Remaining features:")
print([
    'ctr',
    'avg_position',
    'impressions_7d_avg',
    'engagement_per_session',
    'scroll_per_session'
])

Leaky feature removed.
Remaining features:
['ctr', 'avg_position', 'impressions_7d_avg', 'engagement_per_session', 'scroll_per_session']


In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd

HF_TOKEN = userdata.get('HF_TOKEN')

# First, just explore what files exist in the dataset
from huggingface_hub import list_repo_files
files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=HF_TOKEN)
print(files)

['.gitattributes', 'README.md', 'dim_clients.parquet', 'dim_content.parquet', 'fact_content_daily_performance/month=2025-01/data_0.parquet', 'fact_content_daily_performance/month=2025-02/data_0.parquet', 'fact_content_daily_performance/month=2025-03/data_0.parquet', 'fact_content_daily_performance/month=2025-04/data_0.parquet', 'fact_content_daily_performance/month=2025-05/data_0.parquet', 'fact_content_daily_performance/month=2025-06/data_0.parquet', 'fact_content_daily_performance/month=2025-07/data_0.parquet', 'fact_content_daily_performance/month=2025-08/data_0.parquet', 'fact_content_daily_performance/month=2025-09/data_0.parquet', 'fact_content_daily_performance/month=2025-10/data_0.parquet', 'fact_content_daily_performance/month=2025-11/data_0.parquet', 'fact_content_daily_performance/month=2025-12/data_0.parquet', 'fact_content_daily_performance/month=2026-01/data_0.parquet', 'fact_content_daily_performance/month=2026-02/data_0.parquet', 'fact_content_daily_performance/month=20

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation: This analysis uses only March 2026 data, so seasonal patterns
and unusual events from other months are not captured.
The features should be tested across a wider time period
before being considered reliable for other months.

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.